In [1]:
import pandas as pd
import numpy as np
import os
import warnings
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

In [2]:
path = r"C:\Users\Asus\Desktop\TrainDAP\data.csv"
df = pd.read_csv(path)
df.columns = df.columns.str.strip()

target = df.columns[-1]
X = df.drop(columns=[target])
y = df[target]

# Mã hóa nhãn
le_init = LabelEncoder()
for col in X.select_dtypes(include=['object']).columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))
y = le_init.fit_transform(y.astype(str))
X = X.fillna(X.median())

# Chia tập dữ liệu an toàn
counts = np.bincount(y)
strat = y if np.min(counts[counts > 0]) > 1 else None
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=strat)

# SMOTE thích ứng
min_train = np.min(np.bincount(y_train)[np.bincount(y_train) > 0])
k_neighbors = min(5, min_train - 1) if min_train > 1 else 1
if min_train > 1:
    X_train_res, y_train_res = SMOTE(random_state=42, k_neighbors=k_neighbors).fit_resample(X_train, y_train)
else:
    X_train_res, y_train_res = X_train, y_train

# Ép nhãn về dãy liên tục [0, 1, 2...] để XGBoost đạt hiệu suất cao nhất
final_le = LabelEncoder()
y_train_final = final_le.fit_transform(y_train_res)
mask = np.isin(y_test, final_le.classes_)
X_test_clean = X_test[mask]; y_test_clean = final_le.transform(y_test[mask])

In [10]:
threshold = 5  # Các lớp có ít hơn 5 mẫu sẽ bị gộp vào nhóm "Other"

# 1. Gộp các lớp hiếm
class_counts = df[target].value_counts()
rare_classes = class_counts[class_counts < threshold].index
df_boosted = df.copy()
df_boosted[target] = df_boosted[target].apply(lambda x: 'Other' if x in rare_classes else x)

print(f"✅ Đã gộp {len(rare_classes)} lớp hiếm vào nhóm 'Other'.")
print(f"📊 Số lượng lớp mới: {df_boosted[target].nunique()}")

# 2. Tiền xử lý lại trên tập dữ liệu đã gộp
X_b = df_boosted.drop(columns=[target])
y_b = df_boosted[target]

# Encode và Fillna
le_b = LabelEncoder()
for col in X_b.select_dtypes(include=['object']).columns:
    X_b[col] = LabelEncoder().fit_transform(X_b[col].astype(str))
y_b = le_b.fit_transform(y_b.astype(str))
X_b = X_b.fillna(X_b.median())

# 3. Chia tập dữ liệu và SMOTE mạnh hơn
X_train, X_test, y_train, y_test = train_test_split(X_b, y_b, test_size=0.2, random_state=42, stratify=y_b)

# Tính toán k_neighbors an toàn
min_samples_train = np.min(np.bincount(y_train))
k_neighbors = min(5, min_samples_train - 1) if min_samples_train > 1 else 1

smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"🚀 Kích thước tập Train sau khi tối ưu: {X_train_res.shape}")

✅ Đã gộp 5 lớp hiếm vào nhóm 'Other'.
📊 Số lượng lớp mới: 60
🚀 Kích thước tập Train sau khi tối ưu: (7080, 14)


In [15]:
# ===================== IMPORT =====================
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from imblearn.combine import SMOTETomek

# ===================== 1. LỌC LỚP ÍT MẪU =====================
min_samples = 15

class_counts = df[target].value_counts()
top_classes = class_counts[class_counts >= min_samples].index

df_clean = df[df[target].isin(top_classes)].copy()
print(f"📊 Giữ lại {len(top_classes)} lớp | Tổng mẫu: {len(df_clean)}")

# ===================== 2. TIỀN XỬ LÝ =====================
X = df_clean.drop(columns=[target]).copy()
y = df_clean[target].copy()

# Encode categorical
le_dict = {}
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

# Encode target
y = LabelEncoder().fit_transform(y.astype(str))

# Fill missing
X = X.fillna(X.median(numeric_only=True))

# ===================== 3. LOẠI OUTLIERS =====================
iso = IsolationForest(contamination=0.1, random_state=42)
mask = iso.fit_predict(X) != -1

X, y = X[mask], y[mask]
print(f"🧹 Sau khi lọc nhiễu: {X.shape[0]} mẫu")

# ===================== 4. CÂN BẰNG DỮ LIỆU =====================
smt = SMOTETomek(random_state=42)
X_res, y_res = smt.fit_resample(X, y)

print(f"⚖️ Sau cân bằng: {X_res.shape[0]} mẫu")

# ===================== 5. TRAIN MODEL =====================
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res
)

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1  # tận dụng đa lõi CPU
)

print("⏳ Training...")
model.fit(X_train, y_train)

# ===================== 6. EVALUATION =====================
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\n🎯 ACCURACY: {acc*100:.2f}%")

📊 Giữ lại 47 lớp | Tổng mẫu: 2171
🧹 Sau khi lọc nhiễu: 1954 mẫu
⚖️ Sau cân bằng: 5260 mẫu
⏳ Training...

🎯 ACCURACY: 77.28%
